# Lab 8｜活動公告檢查官：Reflection Agent

**陽明交大 113611097 百川 江晟維**  
**方法：Reflection（自我檢查與修訂）**

## 1. 任務與設計

使用者輸入活動名稱、時間、地點、費用與補充資訊，Agent 產生一則可發布的繁體中文活動公告。它不是一次直接回答，而是走三次模型呼叫：

1. **撰稿**：寫初稿。
2. **反思**：另一段審稿 prompt 檢查遺漏、憑空添加與可讀性，提出具體修改。
3. **修訂**：把初稿與審稿意見交回模型改稿。

程式另逐字查核四個必填欄位。若修訂稿仍漏欄位，程式會把原始資訊補在公告尾端。**逐字查核只能確認字串出現，不能取代 AI 的語意審稿。**

這是我自己設計的活動公告主題與流程；本機沒有老師的 Demo07 原始檔，因此沒有直接複製其程式。


## 2. 安裝與模型設定

Colab 左側 **🔑 密鑰** 新增名稱 `Groq` 的金鑰，並開啟「筆記本存取權」。執行時不會印出金鑰。若在本機執行而沒有金鑰，程式會使用已啟動的 Ollama `qwen2.5:14b`。

In [ ]:
%pip install -q "gradio==6.30.0"


## 3. Agent 核心程式

重點看 `run_reflection()`：`draft`、`critique`、`revised` 是三次不同的模型輸出；審稿意見確實送進第三次呼叫。`check_facts()` 顯示修訂前後的客觀欄位檢查。

In [ ]:
%%writefile agent.py
"""活動公告檢查官：初稿 → AI 反思 → 修訂，外加可重現的欄位查核。"""

from __future__ import annotations

import json
import os
import sys
import urllib.error
import urllib.request
from dataclasses import dataclass
from typing import Callable


GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
OLLAMA_URL = "http://127.0.0.1:11434/api/chat"
DEFAULT_GROQ_MODEL = "openai/gpt-oss-120b"
DEFAULT_OLLAMA_MODEL = "qwen2.5:14b"


def _colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        return userdata.get(name)
    except Exception:
        return None


class ChatModel:
    """優先用 Colab 的 Groq 金鑰；本機無金鑰時使用 Ollama。"""

    def __init__(self) -> None:
        self.api_key = (os.getenv("GROQ_API_KEY") or _colab_secret("Groq") or "").strip()
        self.backend = "Groq" if self.api_key else "Ollama"
        self.model = os.getenv("LAB8_MODEL") or (
            DEFAULT_GROQ_MODEL if self.api_key else DEFAULT_OLLAMA_MODEL
        )
        if not self.api_key:
            if "google.colab" in sys.modules:
                raise RuntimeError(
                    "Colab 讀不到 Groq API 金鑰。請在左側 🔑 密鑰新增 Name=Groq，"
                    "Value=Groq 官方網站建立的 API 金鑰，並打開這份筆記本的 Notebook access；"
                    "之後重新執行本格。不要把金鑰貼在程式碼或聊天室。"
                )
            try:
                urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).close()
            except (urllib.error.URLError, TimeoutError) as exc:
                raise RuntimeError(
                    "找不到模型：請在 Colab 左側 🔑 設定 Groq 金鑰並開啟筆記本存取權；"
                    "或在本機啟動 Ollama 並下載 qwen2.5:14b。"
                ) from exc

    def complete(self, system: str, user: str, temperature: float = 0.3) -> str:
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        if self.api_key:
            url = GROQ_URL
            payload = {
                "model": self.model,
                "messages": messages,
                "temperature": temperature,
                "max_tokens": 900,
            }
            # Groq 前端會擋掉 urllib 預設的 Python-urllib User-Agent（403/1010）。
            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "User-Agent": "taica-lab8-reflection-agent/1.0",
            }
        else:
            url = OLLAMA_URL
            payload = {
                "model": self.model,
                "messages": messages,
                "stream": False,
                "options": {"temperature": temperature, "num_predict": 900},
            }
            headers = {}
        request = urllib.request.Request(
            url,
            data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
            headers={"Content-Type": "application/json", **headers},
            method="POST",
        )
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                data = json.load(response)
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")[:300]
            raise RuntimeError(f"{self.backend} 呼叫失敗（HTTP {exc.code}）：{detail}") from exc
        except (urllib.error.URLError, TimeoutError) as exc:
            raise RuntimeError(f"{self.backend} 連線失敗：{exc}") from exc
        answer = (
            data["choices"][0]["message"]["content"]
            if self.api_key
            else data["message"]["content"]
        )
        if not isinstance(answer, str) or not answer.strip():
            raise RuntimeError("模型沒有回傳文字。")
        return answer.strip()


@dataclass(frozen=True)
class Activity:
    title: str
    when: str
    where: str
    fee: str
    notes: str = ""

    def __post_init__(self) -> None:
        for label, value in self.required_facts().items():
            if not value or not value.strip():
                raise ValueError(f"請填寫「{label}」。")
            if len(value) > 120:
                raise ValueError(f"「{label}」請勿超過 120 字。")
        if len(self.notes) > 500:
            raise ValueError("補充資訊請勿超過 500 字。")

    def required_facts(self) -> dict[str, str]:
        return {
            "活動名稱": self.title.strip(),
            "時間": self.when.strip(),
            "地點": self.where.strip(),
            "費用": self.fee.strip(),
        }

    def as_json(self) -> str:
        return json.dumps(
            {**self.required_facts(), "補充資訊": self.notes.strip()},
            ensure_ascii=False,
            indent=2,
        )


def check_facts(text: str, activity: Activity) -> dict[str, bool]:
    """只檢查四個必填值是否逐字出現；不宣稱能判斷語意真偽。"""
    return {label: value in text for label, value in activity.required_facts().items()}


@dataclass(frozen=True)
class ReflectionResult:
    draft: str
    critique: str
    revised: str
    final: str
    before: dict[str, bool]
    after: dict[str, bool]
    repaired: tuple[str, ...]


def run_reflection(
    activity: Activity,
    complete: Callable[[str, str, float], str],
) -> ReflectionResult:
    """三次模型呼叫分別扮演撰稿、審稿、改稿；最後用程式補齊必填資訊。"""
    facts = activity.as_json()
    draft = complete(
        "你是大學活動文案編輯。只寫可直接發布的繁體中文公告，不要說明過程。"
        "只可使用提供的事實，不可編造報名連結、主辦單位、名額或截止日。"
        "請保留活動名稱、時間、地點、費用的原始字串。",
        f"請根據以下資料寫一則 100～220 字、語氣親切而清楚的活動公告：\n{facts}",
        0.6,
    )
    before = check_facts(draft, activity)
    missing_before = [label for label, present in before.items() if not present]
    critique = complete(
        "你是嚴格但具建設性的文案審稿人。請檢查初稿是否漏掉必填資訊、"
        "是否自行加入未提供的事實，以及讀者能否一眼看懂。"
        "請用繁體中文列出『保留』與『修改』兩段；每個修改點要具體。"
        "你只審稿，不要直接重寫公告。",
        f"原始資料：\n{facts}\n\n初稿：\n{draft}\n\n"
        f"程式逐字查核發現缺少的欄位：{missing_before or '無'}。"
        "逐字查核僅供參考；也請檢查語意與憑空添加的內容。",
        0.2,
    )
    revised = complete(
        "你是同一位文案編輯。根據審稿意見重寫公告，只輸出可直接發布的繁體中文修訂稿。"
        "必須逐字保留活動名稱、時間、地點、費用；不可捏造任何資料。",
        f"原始資料：\n{facts}\n\n初稿：\n{draft}\n\n審稿意見：\n{critique}\n\n"
        f"初稿逐字查核缺漏：{missing_before or '無'}。",
        0.3,
    )
    after = check_facts(revised, activity)
    missing_after = [label for label, present in after.items() if not present]
    final = revised
    if missing_after:
        # 模型可能改寫日期或省略費用；程式確保繳出的公告仍含原始必填資訊。
        footer = "\n\n活動資訊：\n" + "\n".join(
            f"- {label}：{activity.required_facts()[label]}" for label in missing_after
        )
        final += footer
    return ReflectionResult(
        draft=draft,
        critique=critique,
        revised=revised,
        final=final,
        before=before,
        after=check_facts(final, activity),
        repaired=tuple(missing_after),
    )


def render_check_table(result: ReflectionResult) -> str:
    rows = ["| 必填資訊 | 初稿 | 最終稿 |", "|---|:---:|:---:|"]
    for label in result.before:
        rows.append(
            f"| {label} | {'✓' if result.before[label] else '✗'} | "
            f"{'✓' if result.after[label] else '✗'} |"
        )
    note = (
        "程式補齊：" + "、".join(result.repaired) + "。"
        if result.repaired
        else "修訂稿已包含所有必填欄位。"
    )
    return "\n".join(rows) + "\n\n" + note + "\n\n逐字查核只驗證欄位字串；AI 審稿另檢查語意與多餘資訊。"


## 4. 可重現的小測試

這裡用固定回覆模擬模型漏掉時間、修訂稿又漏掉費用。測試不用金鑰，能確認三個階段依序執行、審稿意見被帶入改稿、程式補齊缺漏。

In [ ]:
%%writefile test_agent.py
"""可重現的邊界測試：模型漏資訊時，程式仍能補齊必填欄位。"""

import unittest

from agent import Activity, check_facts, run_reflection


class ReflectionTests(unittest.TestCase):
    def test_three_distinct_calls_and_missing_fact_repair(self):
        replies = iter(
            [
                "桌遊夜來了！地點在圖書館 B1 討論室，免費參加。",
                "保留：語氣友善。修改：缺少時間，也沒有完整活動名稱。",
                "期中考後解壓桌遊夜將在 11/15 18:30 舉行，地點為圖書館 B1 討論室。",
            ]
        )
        calls = []

        def fake_model(system, user, temperature):
            calls.append((system, user, temperature))
            return next(replies)

        activity = Activity(
            "期中考後解壓桌遊夜", "11/15 18:30", "圖書館 B1 討論室", "免費"
        )
        result = run_reflection(activity, fake_model)
        self.assertEqual(len(calls), 3)
        self.assertFalse(result.before["時間"])
        self.assertIn("時間", calls[1][1])
        self.assertIn("缺少時間", calls[2][1])
        self.assertEqual(result.repaired, ("費用",))
        self.assertTrue(all(check_facts(result.final, activity).values()))
        self.assertIn("- 費用：免費", result.final)

    def test_empty_required_field_is_rejected(self):
        with self.assertRaisesRegex(ValueError, "費用"):
            Activity("活動", "11/15", "教室", " ")


if __name__ == "__main__":
    unittest.main()


In [ ]:
!python -m unittest -v test_agent.py


## 5. 實際模型測試

這格會真的呼叫三次模型，列出初稿、AI 反思、修訂稿及欄位查核。結果會依模型回覆略有不同。

In [ ]:
from agent import Activity, ChatModel, run_reflection, render_check_table
from IPython.display import display, Markdown

sample = Activity(
    "期中考後解壓桌遊夜",
    "11/15（週日）18:30–21:00",
    "圖書館 B1 討論室",
    "免費",
    "自備水杯；歡迎第一次接觸桌遊的同學。",
)
model = ChatModel()
result = run_reflection(sample, model.complete)
display(Markdown("### ① 初稿\n" + result.draft))
display(Markdown("### ② AI 反思\n" + result.critique))
display(Markdown("### ③ 最終公告\n" + result.final))
display(Markdown("### ④ 程式查核\n" + render_check_table(result)))


## 6. Gradio 展示與對話結果

點「產生並反思公告」可看到對話中的最終公告，以及可展開的初稿、反思、修訂稿。可改掉活動資料再執行，觀察 Agent 是否保留原始資訊。Colab 啟動時會產生 Gradio 分享網址（只在這次執行期間有效）。

以下是本機同一份 `app.py` 的實際操作截圖；可看見對話、欄位查核和三階段文字。完整文字亦保存在專案的 `screenshots/gradio_result.json`。

![Gradio 對話結果](https://raw.githubusercontent.com/arvin611ch/taica-lab8-reflection-agent/main/screenshots/gradio_chat.png)

![初稿、AI 反思與修訂稿](https://raw.githubusercontent.com/arvin611ch/taica-lab8-reflection-agent/main/screenshots/gradio_full.png)


In [ ]:
%%writefile app.py
"""Gradio 介面；可本機執行，也可在 Colab 最後一格啟動。"""

import gradio as gr

from agent import Activity, ChatModel, render_check_table, run_reflection


def respond(title, when, where, fee, notes, history):
    history = list(history or [])
    try:
        activity = Activity(title, when, where, fee, notes)
        model = ChatModel()
        result = run_reflection(activity, model.complete)
    except (ValueError, RuntimeError) as exc:
        return history, "", "", "", f"⚠️ {exc}"

    summary = f"{title}｜{when}｜{where}｜{fee}"
    history.append({"role": "user", "content": summary})
    history.append(
        {
            "role": "assistant",
            "content": f"### 完成公告\n{result.final}\n\n"
            f"_已執行：初稿 → AI 反思 → 修訂；模型：{model.backend}／{model.model}_",
        }
    )
    return history, result.draft, result.critique, result.revised, render_check_table(result)


with gr.Blocks(title="活動公告檢查官｜Reflection Agent") as demo:
    gr.Markdown(
        "# 活動公告檢查官 ✍️\n"
        "**Reflection Agent**：撰稿 AI → 審稿 AI → 改稿 AI。"
        "下方會公開三個階段與欄位查核，方便觀察修訂是否真的有幫助。"
    )
    with gr.Row():
        with gr.Column(scale=1):
            title = gr.Textbox(label="活動名稱（必填）", value="期中考後解壓桌遊夜")
            when = gr.Textbox(label="時間（必填）", value="11/15（週日）18:30–21:00")
            where = gr.Textbox(label="地點（必填）", value="圖書館 B1 討論室")
            fee = gr.Textbox(label="費用（必填）", value="免費")
            notes = gr.Textbox(
                label="補充資訊（選填）",
                value="自備水杯；歡迎第一次接觸桌遊的同學。",
                lines=3,
            )
            run_button = gr.Button("產生並反思公告", variant="primary")
            gr.Markdown("Colab：左側 🔑 新增 `Groq` 金鑰並開啟筆記本存取權。")
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Gradio 對話結果", height=400)
            checks = gr.Markdown(label="必填資訊查核")
    with gr.Accordion("查看 Agent 的三個階段", open=True):
        draft_box = gr.Textbox(label="① 初稿", lines=6, interactive=False)
        critique_box = gr.Textbox(label="② AI 反思／審稿意見", lines=8, interactive=False)
        revised_box = gr.Textbox(label="③ AI 修訂稿（程式補齊前）", lines=6, interactive=False)
    run_button.click(
        respond,
        inputs=[title, when, where, fee, notes, chatbot],
        outputs=[chatbot, draft_box, critique_box, revised_box, checks],
    )
    gr.ClearButton([chatbot, draft_box, critique_box, revised_box, checks])


if __name__ == "__main__":
    demo.launch(share=False)


In [ ]:
from app import demo
demo.launch(share=True)


## 7. 改良與限制

- 審稿 prompt 被限定只能提出具體修改，改稿 prompt 明確接收審稿內容，避免把 Reflection 寫成三次彼此無關的生成。
- 最終公告附有程式逐字查核與缺漏補齊，能觀察模型有沒有落實修改。
- 逐字查核無法判斷活動資訊是否被錯誤解釋；例如日期換成等價寫法仍可能被標成缺漏。AI 審稿可補充語意判斷，但也可能看漏，因此畫面保留三階段供人工檢視。
- 若模型服務無法連線，介面顯示錯誤，不會把預設文案冒充 AI 執行結果。